# Run the Cancer Myth RAG over every question

This notebook classifies the first `N_QUESTIONS` records in `data/cancermyth_screening_dataset.json` with the existing GPT-5.6 Luna RAG pipeline. It writes `answers.csv` with exactly two columns: `question_id` and `answer`.

Set `OPENAI_API_KEY` in the environment and choose `N_QUESTIONS` before running all cells. A CPU-sized worker pool classifies questions concurrently. Results are appended and flushed after every question, so rerunning the batch resumes from the existing CSV without repeating completed API calls. Delete `answers.csv` only when you intentionally want to start over.

In [ ]:
from __future__ import annotations

from concurrent.futures import as_completed, ThreadPoolExecutor
import csv
import json
import os
from pathlib import Path
import time


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root from the notebook directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"
with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    questions = json.load(dataset_file)
PROMPT_KEY = "basic"
OUTPUT_PATH = PROJECT_ROOT / "rag" / "rag_run_all" / f"answers_{PROMPT_KEY}.csv"
TOP_K = 6
MAX_ATTEMPTS = 3
N_QUESTIONS = len(questions)  # Run only the first N questions in the JSON dataset.
N_WORKERS = max(1, os.cpu_count() or 4)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output CSV: {OUTPUT_PATH}")

Project root: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth
Output CSV: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\rag\rag_run_all\answers.csv


In [14]:
if not isinstance(questions, list) or not questions:
    raise ValueError("The dataset must be a non-empty JSON array.")

required_fields = {"id", "question"}
for index, record in enumerate(questions):
    if not isinstance(record, dict) or not required_fields.issubset(record):
        raise ValueError(f"Dataset item {index} is missing 'id' or 'question'.")
    if not str(record["question"]).strip():
        raise ValueError(f"Dataset item {index} has an empty question.")

question_ids = [str(record["id"]) for record in questions]
if len(question_ids) != len(set(question_ids)):
    raise ValueError("Question IDs must be unique.")
if isinstance(N_QUESTIONS, bool) or not isinstance(N_QUESTIONS, int):
    raise TypeError("N_QUESTIONS must be an integer.")
if not 1 <= N_QUESTIONS <= len(questions):
    raise ValueError(f"N_QUESTIONS must be between 1 and {len(questions)}.")

selected_questions = questions[:N_QUESTIONS]

print(f"Loaded {len(questions)} questions; selected the first {len(selected_questions)}.")
print(f"Concurrent workers: {min(N_WORKERS, len(selected_questions))}")

Loaded 735 questions; selected the first 735.
Concurrent workers: 16


In [15]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.rag_model.config import (
    LLMSettings,
    PDQ_CANCERS_DIR,
    PDQ_GENERAL_TOPICS_DIR,
)
from rag.rag_model.corpus import load_pdq_chunks
from rag.rag_model.retriever import BM25Retriever
from rag.rag_model.service import CancerMythRAG

chunks = load_pdq_chunks((PDQ_CANCERS_DIR, PDQ_GENERAL_TOPICS_DIR))
rag = CancerMythRAG(BM25Retriever(chunks), top_k=TOP_K)
print(f"Indexed {len(chunks)} NCI PDQ chunks.")

Indexed 20596 NCI PDQ chunks.


In [16]:
settings = LLMSettings.from_environment()
settings.validate()
if settings.base_url.rstrip("/") == "https://api.openai.com/v1" and not settings.api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running the batch.")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
csv_columns = ["question_id", "answer"]
completed_ids: set[str] = set()

if OUTPUT_PATH.exists() and OUTPUT_PATH.stat().st_size:
    with OUTPUT_PATH.open(newline="", encoding="utf-8") as existing_file:
        reader = csv.DictReader(existing_file)
        if reader.fieldnames != csv_columns:
            raise ValueError(f"Existing CSV must have columns {csv_columns}; got {reader.fieldnames}.")
        for row in reader:
            if row["answer"] not in {"true", "false"}:
                raise ValueError(f"Invalid saved answer for question {row['question_id']}.")
            completed_ids.add(row["question_id"])

pending = [record for record in selected_questions if str(record["id"]) not in completed_ids]

print(f"Model: {settings.model}")
completed_selected = sum(str(record["id"]) in completed_ids for record in selected_questions)
worker_count = min(N_WORKERS, max(1, len(pending)))
print(f"Selected: {len(selected_questions)}; already completed: {completed_selected}; running now: {len(pending)}")
print(f"Concurrent workers for this run: {worker_count}")


def classify_with_retry(question: str) -> str:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return rag.classify(question, PROMPT_KEY, settings).text
        except Exception:
            if attempt == MAX_ATTEMPTS:
                raise
            delay_seconds = 2 ** (attempt - 1)
            print(f"Request failed; retrying in {delay_seconds}s (attempt {attempt + 1}/{MAX_ATTEMPTS}).")
            time.sleep(delay_seconds)
    raise AssertionError("Unreachable")


write_header = not OUTPUT_PATH.exists() or OUTPUT_PATH.stat().st_size == 0
with OUTPUT_PATH.open("a", newline="", encoding="utf-8") as results_file:
    writer = csv.DictWriter(results_file, fieldnames=csv_columns)
    if write_header:
        writer.writeheader()
        results_file.flush()
        os.fsync(results_file.fileno())

    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        future_to_record = {
            executor.submit(classify_with_retry, str(record["question"])): record
            for record in pending
        }
        for batch_index, future in enumerate(as_completed(future_to_record), start=1):
            record = future_to_record[future]
            answer = future.result()
            writer.writerow({"question_id": record["id"], "answer": answer})
            results_file.flush()
            os.fsync(results_file.fileno())
            if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(pending):
                print(f"Saved {batch_index}/{len(pending)} pending answers (question_id={record['id']}).")

print(f"Batch complete. Results saved to {OUTPUT_PATH}")

Model: gpt-5.6-luna
Selected: 735; already completed: 100; running now: 635
Concurrent workers for this run: 16
Saved 1/635 pending answers (question_id=101).
Saved 10/635 pending answers (question_id=110).
Saved 20/635 pending answers (question_id=120).
Saved 30/635 pending answers (question_id=123).
Saved 40/635 pending answers (question_id=147).
Saved 50/635 pending answers (question_id=151).
Saved 60/635 pending answers (question_id=162).
Saved 70/635 pending answers (question_id=169).
Saved 80/635 pending answers (question_id=182).
Saved 90/635 pending answers (question_id=189).
Saved 100/635 pending answers (question_id=201).
Saved 110/635 pending answers (question_id=213).
Saved 120/635 pending answers (question_id=219).
Saved 130/635 pending answers (question_id=229).
Saved 140/635 pending answers (question_id=248).
Saved 150/635 pending answers (question_id=245).
Saved 160/635 pending answers (question_id=263).
Saved 170/635 pending answers (question_id=258).
Saved 180/635 pen

In [17]:
with OUTPUT_PATH.open(newline="", encoding="utf-8") as results_file:
    saved_answers = list(csv.DictReader(results_file))

saved_ids = {row["question_id"] for row in saved_answers}
selected_ids = {str(record["id"]) for record in selected_questions}
missing_ids = selected_ids - saved_ids

print(f"Saved selected rows: {len(selected_ids - missing_ids)} / {len(selected_questions)}")
print(f"Total rows currently in CSV: {len(saved_answers)}")
print(saved_answers[:5])

assert not missing_ids, f"The selected run is incomplete; missing question IDs: {sorted(missing_ids)}"

Saved selected rows: 735 / 735
Total rows currently in CSV: 735
[{'question_id': '9', 'answer': 'true'}, {'question_id': '4', 'answer': 'true'}, {'question_id': '5', 'answer': 'true'}, {'question_id': '2', 'answer': 'true'}, {'question_id': '3', 'answer': 'true'}]
